# 06c — 第六讲基础习题参考答案（Solutions）

本 Notebook 给出 `06c_exercises.ipynb` 中 8 道基础编程练习的完整参考解答与讲解。

---

## 0. 公共环境准备

In [ ]:
import os
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from scipy.spatial import ConvexHull

warnings.filterwarnings("ignore")

from pymatgen.core import Structure, Lattice, Composition, Element, Species
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
from pymatgen.io.cif import CifParser
from pymatgen.analysis.diffraction.xrd import XRDCalculator
from pymatgen.entries.computed_entries import ComputedEntry
from pymatgen.analysis.phase_diagram import PhaseDiagram

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

plt.rcParams["figure.dpi"] = 110
np.random.seed(42)

def find_file(name):
    for path in (f"../data/{name}", f"data/{name}", name):
        if os.path.exists(path):
            return path
    raise FileNotFoundError(f"找不到 {name}")

xrd = XRDCalculator(wavelength="CuKa")

<!-- ai4m-style -->
> **📘 课程统一绘图风格**：下面这格加载课程自带的 `ai4m.mplstyle`（配色与课程网站同源，无上右边框，仅横向浅网格，保存默认 300 dpi）。`C` 是课程配色，手动指定颜色时写 `color=C.crimson`。


In [ ]:
# ai4m-style：课程统一绘图风格（solutions/ 子目录与 Colab 通用）
import pathlib, sys

for _p in (pathlib.Path.cwd().parent, pathlib.Path.cwd().parent.parent):
    sys.path.insert(0, str(_p))   # course/ 根目录
import ai4m_style as ai4m

C = ai4m.use()        # 应用 ai4m.mplstyle，返回课程配色


---

## 第一部分：晶体结构与对称性

### 习题 1 参考答案：搭建闪锌矿 GaAs

In [ ]:
gaas = Structure.from_spacegroup(
    "F-43m", Lattice.cubic(5.653), ["Ga", "As"],
    [[0, 0, 0], [0.25, 0.25, 0.25]])

print(f"化学式       : {gaas.composition.reduced_formula}")
print(f"晶胞原子数   : {gaas.num_sites}")
print(f"晶胞体积     : {gaas.volume:.2f} A^3")
print(f"理论密度     : {gaas.density:.3f} g/cm^3   (实验值 5.32)")

# 关键：对所有 As 取最小值，而不是随手挑两个编号
i_ga = next(i for i, s in enumerate(gaas.sites) if s.specie.symbol == "Ga")
idx_as = [i for i, s in enumerate(gaas.sites) if s.specie.symbol == "As"]
dists = sorted(gaas.get_distance(i_ga, j) for j in idx_as)
print(f"\nGa(0) 到四个 As 的距离: {[round(d, 4) for d in dists]} A")
print(f"最近邻 Ga-As 键长: {dists[0]:.4f} A")
print(f"解析值 sqrt(3)*a/4 = {3 ** 0.5 * 5.653 / 4:.4f} A")

**讲解**：闪锌矿结构里 As 位于 Ga 构成的四面体空隙中心，最近邻距离正是体对角线的四分之一：

$$d_{\text{Ga-As}} = \frac{\sqrt{3}}{4}\,a = \frac{1.732 \times 5.653}{4} = 2.448\ \text{Å}$$

数值完全吻合，密度 5.318 g/cm³ 也与实验值 5.32 对得上。

注意打印出来的四个距离：三个是 2.448 Å，还有一个是 4.90 Å。**如果随手写 `get_distance(0, 4)`，拿到的很可能就是那个 4.90**——这正是 `06a` 第 1 节演示过的坑。周期性边界下，位点编号的先后和空间距离毫无关系。

闪锌矿与金刚石结构的几何完全相同（两套互相穿插的 fcc 点阵），区别只在于金刚石的两套点阵放同种原子，闪锌矿放两种。正因如此，Si 有反演中心而 GaAs 没有——**GaAs 的压电性与二阶非线性光学效应，根源就在这个对称性差别上**。

### 习题 2 参考答案：对称性分析与原始胞

In [ ]:
sga = SpacegroupAnalyzer(gaas, symprec=0.01)

print(f"空间群符号   : {sga.get_space_group_symbol()}")
print(f"空间群编号   : {sga.get_space_group_number()}")
print(f"晶系         : {sga.get_crystal_system()}")
print(f"点群         : {sga.get_point_group_symbol()}")
print(f"对称操作数   : {len(sga.get_symmetry_operations())}")

prim = sga.get_primitive_standard_structure()
print(f"\n常规胞: {gaas.num_sites} 个原子, V = {gaas.volume:.2f} A^3")
print(f"原始胞: {prim.num_sites} 个原子, V = {prim.volume:.2f} A^3")
print(f"原子数之比: {gaas.num_sites / prim.num_sites:.0f}")
print(f"\n按 O(N^3) 估计，用原始胞做 DFT 的计算量约为常规胞的 "
      f"{(prim.num_sites / gaas.num_sites) ** 3:.2%}")

**讲解**：`F-43m`（No. 216）是面心立方点阵、点群 $\bar{4}3m$。注意这个点群**没有反演中心**——这正是 GaAs 与金刚石结构 Si（$Fd\bar{3}m$，有反演中心）的关键差别。

原始胞只有 2 个原子（1 个 Ga + 1 个 As），而常规立方胞有 8 个。两者描述的是**同一个晶体**，但计算量差了 $(2/8)^3 \approx 1.6\%$——也就是说用原始胞快约 64 倍。

> **📌 实战习惯**：拿到任何结构，先 `SpacegroupAnalyzer(s).get_primitive_standard_structure()` 化到原始胞再提交 DFT 计算，这是免费的几十倍加速。唯一的例外是要研究缺陷、表面或超晶格时——那时你**需要**打破对称性，就得老老实实用大胞（见练习 06a-2）。

### 习题 3 参考答案：Goldschmidt 容差因子

In [ ]:
R = {"Cs": 1.88, "Rb": 1.72, "K": 1.64, "Pb": 1.19, "I": 2.20}

rows = []
for A in ["Cs", "Rb", "K"]:
    t = (R[A] + R["I"]) / (np.sqrt(2) * (R["Pb"] + R["I"]))
    rows.append({"perovskite": f"{A}PbI3", "r_A": R[A], "r_B": R["Pb"],
                 "r_X": R["I"], "t": round(t, 4),
                 "in_window": bool(0.8 < t < 1.0)})

tol = pd.DataFrame(rows)
print(tol.to_string(index=False))

fig, ax = plt.subplots(figsize=(7.4, 3.4))
ax.barh(tol.perovskite, tol.t, color=[C.navy, C.gold, C.crimson], height=0.55)
ax.axvspan(0.8, 1.0, color=C.gold, alpha=0.15, label="Goldschmidt 窗口")
ax.axvline(0.8, color=C.crimson, ls="--", lw=1.4)
for i, t in enumerate(tol.t):
    ax.text(t + 0.004, i, f"{t:.3f}", va="center", fontsize=9)
ax.set_xlim(0.75, 1.02)
ax.set_xlabel("tolerance factor t")
ax.set_title("三个碘铅钙钛矿的容差因子", fontsize=11)
ax.legend(fontsize=9, loc="lower right")
plt.show()

**讲解**：三个值依次是 $0.851$、$0.818$、$0.801$——随 A 位离子变小而单调下降，完全符合公式里 $r_A$ 出现在分子上的结构。

**但注意第 3 问的关键**：三个值**全都落在 $0.8 < t < 1.0$ 的窗口里**，包括实验上根本做不出钙钛矿的 $\text{KPbI}_3$（它实际形成的是一维链状的非钙钛矿结构）。容差因子并没有把它排除掉，只是让它紧贴下边界。

这恰恰印证了 `06b` 第 5 节在 1,929 条 DFT 数据上得到的结论：**$t$ 是窗口型判据，分辨力有限**。

**主要产生哪一类错误？假阳性。** 因为 $t$ 只用了三个离子半径，完全不含电子结构信息，它会把一大批“几何上装得下、化学上不成立”的组成放进来。所以在高通量筛选里，$t$ 适合做**第一道粗筛**（把 $t < 0.7$ 或 $t > 1.1$ 的明显不可能组成快速排除，这一步极便宜），但**绝不能单独用它下结论**——后面必须跟上 DFT 的 $E_{\text{hull}}$ 验证。

顺带一提，近年文献提出了改进判据（如 Bartel 等人 2019 年的 $\tau$ 因子），加入了 A 位氧化态与半径比的非线性项，分类准确率明显高于经典的 $t$。

---

## 第二部分：XRD 与热力学

### 习题 4 参考答案：模拟 XRD 并反算 d 值

In [ ]:
pattern = xrd.get_pattern(gaas, two_theta_range=(20, 80))

print(f"{'2theta/deg':>11s} {'d/A':>9s} {'I_rel':>8s}   (hkl)")
for tt, inten, hg, d in list(zip(pattern.x, pattern.y, pattern.hkls, pattern.d_hkls))[:5]:
    print(f"{tt:11.3f} {d:9.4f} {inten:8.1f}   {hg[0]['hkl']}")

k = int(np.argmax(pattern.y))
lam = xrd.wavelength
theta = np.radians(pattern.x[k] / 2)
d_manual = lam / (2 * np.sin(theta))

print(f"\n最强峰: 2theta = {pattern.x[k]:.3f} deg, (hkl) = {pattern.hkls[k][0]['hkl']}")
print(f"  手工用布拉格定律反算 d = lambda / (2 sin theta) = {d_manual:.4f} A")
print(f"  pymatgen 给出的 d_hkl                        = {pattern.d_hkls[k]:.4f} A")
print(f"  差值 = {abs(d_manual - pattern.d_hkls[k]):.2e} A    (波长用的是 {lam} A)")

fig, ax = plt.subplots(figsize=(9, 3.8))
ax.vlines(pattern.x, 0, pattern.y, color=C.navy, lw=2.2)
for tt, inten, hg in zip(pattern.x, pattern.y, pattern.hkls):
    if inten > 5:
        ax.text(tt, inten + 2.5, "".join(str(abs(v)) for v in hg[0]["hkl"]),
                ha="center", fontsize=8)
ax.set_xlabel("2theta / degree  (Cu K-alpha)")
ax.set_ylabel("I (rel.)")
ax.set_ylim(0, 115)
ax.set_title("Simulated XRD of zinc-blende GaAs")
plt.show()

**讲解**：手工反算与 pymatgen 的结果在机器精度内一致——因为 `XRDCalculator` 内部就是先由晶格算 $d_{hkl}$、再用布拉格定律换算成 $2\theta$，我们只是把这个过程倒着走了一遍。

**唯一容易出错的地方是波长。** `XRDCalculator(wavelength="CuKa")` 用的是 Cu K$\alpha_1$ 与 K$\alpha_2$ 的**加权平均** 1.54184 Å，而很多教科书写的是 K$\alpha_1$ 的 1.54056 Å。两者差 0.08%，换算出来的 $d$ 在小数点后第三位就会不一致。所以代码里要用 `xrd.wavelength` 取回计算器自己用的那个值，而不是硬编码常数。

最强峰 $(111)$ 位于 $2\theta \approx 27.3^\circ$。注意谱图里 $(200)$ 峰**极弱甚至看不到**——面心立方要求 $h,k,l$ 全奇或全偶，$(200)$ 虽然满足，但 GaAs 的 $(200)$ 强度正比于 $|f_{\text{Ga}} - f_{\text{As}}|^2$，而 Ga（$Z=31$）与 As（$Z=33$）的电子数极为接近，于是这个峰几乎熄灭——和 `06b` 第 2 节里 KCl 的 $(111)$ 峰是完全相同的物理。

### 习题 5 参考答案：体心立方的系统消光

In [ ]:
fe = Structure.from_spacegroup("Im-3m", Lattice.cubic(2.866), ["Fe"], [[0, 0, 0]])
print(f"alpha-Fe: {fe.composition.reduced_formula}, {fe.num_sites} 个原子/晶胞, "
      f"rho = {fe.density:.3f} g/cm^3  (实验值 7.87)")

p_fe = xrd.get_pattern(fe, two_theta_range=(20, 120))

rows = []
for tt, inten, hg, d in zip(p_fe.x, p_fe.y, p_fe.hkls, p_fe.d_hkls):
    hkl = hg[0]["hkl"]
    rows.append({"2theta": round(float(tt), 2), "hkl": str(hkl),
                 "h+k+l": int(sum(abs(v) for v in hkl)),
                 "parity": "偶" if sum(abs(v) for v in hkl) % 2 == 0 else "奇",
                 "d_A": round(float(d), 4), "I_rel": round(float(inten), 1)})

peaks = pd.DataFrame(rows)
print()
print(peaks.to_string(index=False))
print(f"\n所有峰的 h+k+l 都是偶数吗? {bool((peaks['h+k+l'] % 2 == 0).all())}")

**讲解**：输出里每一个峰的 $h+k+l$ 都是偶数：$(110) \to 2$、$(200) \to 2$、$(211) \to 4$、$(220) \to 4$、$(310) \to 4$……**没有任何一个奇数**。

**为什么？** 体心立方的晶胞里有两个等价原子，分别在 $(0,0,0)$ 和 $(\frac{1}{2},\frac{1}{2},\frac{1}{2})$。结构因子是

$$F_{hkl} = f\left[1 + e^{i\pi(h+k+l)}\right] = \begin{cases} 2f & h+k+l \text{ 为偶数} \\ 0 & h+k+l \text{ 为奇数} \end{cases}$$

体心那个原子散射出的波与角上原子的波**恰好反相**，于是奇数和的衍射全部相消。

最强峰 $(110)$ 在 $2\theta \approx 44.7^\circ$，这是钢铁材料 XRD 里最熟悉的一条线。

**回答第 4 问**：如果谱图里出现了 $(100)$ 峰，说明体心位置的原子与角上的原子**散射因子不同**，此时 $F = f_1 - f_2 \ne 0$。最典型的情况就是**有序合金**：

- **CsCl 型（B2）有序结构**，如 FeAl、NiAl、CuZn（β-黄铜）：角上是 Fe，体心是 Al，于是 $(100)$、$(111)$ 等奇数和的峰出现；
- 这类只在有序相中出现的峰叫**超点阵衍射峰（superlattice reflections）**。

**它的实用价值极大**：通过测量超点阵峰的强度，就能定量得到合金的**长程有序度参数**。无序固溶体里每个位点的平均散射因子相同，超点阵峰消失；随着退火使合金有序化，这些峰逐渐长出来。这是研究有序–无序相变的标准手段。

### 习题 6 参考答案：计算形成能（含元素参考态的坑）

In [ ]:
ea = pd.read_csv(find_file("energy_per_atom_aflow.csv"))
TOKEN = re.compile(r"([A-Z][a-z]?)(\d*\.?\d*)")

def parse_formula(f):
    """把 'O2Ti1' 解析成 {'O': 2.0, 'Ti': 1.0}。"""
    d = {}
    for el, n in TOKEN.findall(f):
        if el:
            d[el] = d.get(el, 0.0) + (float(n) if n else 1.0)
    return d

ea["comp"] = ea.formula.map(parse_formula)
ea["n_el"] = ea.comp.map(len)

elemental = ea[ea.n_el == 1].copy()
elemental["element"] = elemental.comp.map(lambda c: next(iter(c)))

OUTLIER_TOL = 2.0
mu, flagged = {}, []
for el, g in elemental.groupby("element"):
    med = g.energy_per_atom.median()
    dev = (g.energy_per_atom - med).abs()
    ok, bad = g[dev <= OUTLIER_TOL], g[dev > OUTLIER_TOL]
    for f, v in zip(bad.formula, bad.energy_per_atom):
        flagged.append({"element": el, "formula": f,
                        "energy": round(v, 3), "median": round(med, 3)})
    mu[el] = ok.energy_per_atom.min() if len(ok) else med

def formation_energy(comp, e_per_atom):
    total = sum(comp.values())
    if not all(el in mu for el in comp):
        return np.nan
    return e_per_atom - sum(n / total * mu[el] for el, n in comp.items())

ea["E_f"] = [formation_energy(c, v) for c, v in zip(ea.comp, ea.energy_per_atom)]
print(f"{len(mu)} 个元素有参考态；异常条目 {len(flagged)} 条")

print()
print("被判为异常并剔除的元素参考态条目：")
print(pd.DataFrame(flagged).to_string(index=False))

print("\n四个化合物的形成能：")
CHECK = {"O2Ti1": ("TiO2", -3.26), "Al2O3": ("Al2O3", -3.47),
         "O1Zn1": ("ZnO", -1.81), "Cd1Te1": ("CdTe", -0.49)}
for f, (pretty, exp) in CHECK.items():
    row = ea[ea.formula == f]
    if len(row):
        print(f"  {pretty:8s} DFT = {row.E_f.iloc[0]:+.3f}   实验 = {exp:+.2f} eV/atom")

print("\n对照：如果偷懒直接对每个元素取 min（不剔除异常值）会怎样？")
mu_naive = {el: g.energy_per_atom.min() for el, g in elemental.groupby("element")}
for f, pretty in [("S1Zn1", "ZnS"), ("Cu1In1S2", "CuInS2")]:
    row = ea[ea.formula == f]
    if len(row):
        c = row.comp.iloc[0]
        tot = sum(c.values())
        naive = row.energy_per_atom.iloc[0] - sum(n / tot * mu_naive[el]
                                                  for el, n in c.items())
        print(f"  {pretty:8s} 稳健 = {row.E_f.iloc[0]:+8.3f}   "
              f"偷懒 = {naive:+8.3f} eV/atom")

**讲解**：被揪出来的两条异常是 `S8 = -23.559` 和 `Ta8 = +20.750` eV/atom。硫的其他同素异形体都在 $-4.1$ 附近，钽的在 $-9.5$ 附近——这两条显然是数据库里的坏记录（很可能是某次计算没收敛，或者单位/原子数记错）。

四个化合物与实验值的对照：ZnO（$-1.80$ vs $-1.81$）和 CdTe（$-0.49$ vs $-0.49$）几乎精确命中，氧化物偏差 $0.1 \sim 0.5$ eV/atom——这就是 PBE 对形成能的典型精度。

**最后那组对照是本题的重点**：直接取 `min` 的话，ZnS 和 $\text{CuInS}_2$ 的形成能会错掉约 **10 eV/atom**（硫的参考态被 $-23.6$ 那条脏数据占据，每个硫原子的能量被算低约 19.5 eV，按原子分数摊下来就是 10 eV 量级）。

> **📌 这是数据工程里最危险的一类 bug**：它**不会报错**，不抛异常，只会安静地给出一个错了几个数量级的数字。而且因为参考态是共享的，**一条脏数据能污染成千上万条下游结果**。
>
> 防御手段：(1) 对任何“取最值”的聚合操作保持警惕；(2) 用中位数等稳健统计量先定位合理区间；(3) **永远用已知答案做健全性检验**——这里就是拿 ZnO、CdTe 的实验形成能来验。

### 习题 7 参考答案：Cu–O 体系的凸包

In [ ]:
SYSTEM = {"Cu", "O"}
sub = ea[ea.comp.map(lambda c: len(c) > 0 and set(c) <= SYSTEM)].copy()
sub["x_O"] = sub.comp.map(lambda c: c.get("O", 0) / sum(c.values()))
sub["label"] = sub.comp.map(lambda c: Composition(
    "".join(f"{k}{v:g}" for k, v in c.items())).reduced_formula)
print(f"Cu-O 体系共 {len(sub)} 条记录")

best = sub.loc[sub.groupby("x_O").E_f.idxmin()].reset_index(drop=True)
pts = np.column_stack([best.x_O.values, best.E_f.values])
aug = np.vstack([pts, [0.5, pts[:, 1].max() + 10.0]])
hull_idx = sorted(i for i in ConvexHull(aug).vertices if i < len(pts))
hull_pts = pts[hull_idx]

print("\n凸包上的稳定相：")
for i in hull_idx:
    print(f"  x_O = {best.x_O[i]:.4f}   {best.label[i]:8s} "
          f"E_f = {best.E_f[i]:+.4f} eV/atom")

sub["E_hull"] = [ef - np.interp(x, hull_pts[:, 0], hull_pts[:, 1])
                 for x, ef in zip(sub.x_O, sub.E_f)]

print("\n两个实验上已知的稳定氧化物：")
for lbl in ["Cu2O", "CuO"]:
    r = sub[sub.label == lbl]
    if len(r):
        r = r.loc[r.E_f.idxmin()]
        print(f"  {lbl:6s} E_f = {r.E_f:+.4f}   E_hull = {r.E_hull * 1000:6.1f} meV/atom")

fig, ax = plt.subplots(figsize=(8.5, 5))
ax.scatter(sub.x_O, sub.E_f, s=50, color="#b0b0b0", zorder=2, label="全部条目")
ax.plot(hull_pts[:, 0], hull_pts[:, 1], "-o", color=C.crimson, lw=2.0, ms=9,
        zorder=4, label="热力学凸包")
for i in hull_idx:
    ax.annotate(best.label[i], (best.x_O[i], best.E_f[i]),
                textcoords="offset points", xytext=(0, -17), ha="center",
                fontsize=9, color=C.crimson, fontweight="bold")
ax.axhline(0, color="#888888", lw=1.0, ls="--")
ax.set_xlabel("x(O) = O / (Cu + O)")
ax.set_ylabel("Formation energy / eV per atom")
ax.set_title("Cu-O 体系的热力学凸包")
ax.legend(fontsize=9, loc="lower left")
plt.show()

**讲解**：凸包上出现了 **Cu、$\text{Cu}_2\text{O}$、$\text{Cu}_4\text{O}_3$、CuO、$\text{O}_2$**。两个实验上熟知的氧化物 $\text{Cu}_2\text{O}$（赤铜矿）和 CuO（黑铜矿）**都稳稳落在凸包上**（$E_{\text{hull}} = 0$），与实验相图完全一致——和练习 06b-2 里 Fe–O 体系丢掉赤铁矿形成鲜明对比。

**为什么 Cu–O 对了而 Fe–O 错了？** 关键在 $d$ 电子的填充情况：

- $\text{Cu}^+$ 是 $3d^{10}$ 满壳层，$\text{Cu}^{2+}$ 是 $3d^9$——电子相对离域，PBE 的自相互作用误差较小；
- $\text{Fe}^{2+}$（$3d^6$）和 $\text{Fe}^{3+}$（$3d^5$）是部分填充的局域 $d$ 态，还有复杂磁序，PBE 的误差大得多。

一句话：**同一套方法在不同化学体系上的可靠性可以差很远**。这也是为什么 Materials Project 对过渡金属氧化物一律用 DFT+$U$，而且对不同元素取不同的 $U$ 值。

顺带一提，$\text{Cu}_4\text{O}_3$（矿物名 paramelaconite，介于赤铜矿与黑铜矿之间的中间相）在自然界确实存在但制备困难。它出现在凸包上说明 DFT 认为它热力学稳定——这与“实验上难合成”并不矛盾：**热力学稳定说的是终态能量，合成难易说的是动力学路径**。

### 习题 8 参考答案：CBFV 特征与一次诚实的外推测试

In [ ]:
magpie = pd.read_csv(find_file("magpie_element_properties.csv")).set_index("element")
PROPS = list(magpie.columns)
LOOKUP = magpie.to_dict("index")
STATS = ["mean", "avg_dev", "min", "max", "range"]
FEATURE_NAMES = [f"{s}_{p}" for s in STATS for p in PROPS]

def composition_features(comp):
    """22 个元素属性 x 5 个统计量 = 110 维 CBFV。"""
    els = list(comp)
    if not els or not all(e in LOOKUP for e in els):
        return None
    w = np.array([comp[e] for e in els], dtype=float)
    w = w / w.sum()
    M_ = np.array([[LOOKUP[e][p] for p in PROPS] for e in els], dtype=float)
    mean = (w[:, None] * M_).sum(0)
    avg_dev = (w[:, None] * np.abs(M_ - mean)).sum(0)
    return np.concatenate([mean, avg_dev, M_.min(0), M_.max(0), M_.max(0) - M_.min(0)])

bg = (pd.read_csv(find_file("bandgap_aflow.csv"))
        .sample(6000, random_state=0).reset_index(drop=True))
bg["comp"] = bg.formula.map(parse_formula)
bg["elements"] = bg.comp.map(lambda c: frozenset(c))

feats, keep = [], []
for i, c in enumerate(bg.comp):
    v = composition_features(c)
    if v is not None:
        feats.append(v)
        keep.append(i)
X = np.asarray(feats)
bg = bg.iloc[keep].reset_index(drop=True)
y = bg.band_gap.values
print(f"特征矩阵: {X.shape[0]} 个化合物 x {X.shape[1]} 维")

# ① 随机划分
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
m_rand = RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
r2_rand = r2_score(y_te, m_rand.predict(X_te))
print(f"\n① 随机划分        : n_test = {len(y_te):4d}   R2 = {r2_rand:.3f}")

# ② 留出全部氯化物
mask_cl = bg.elements.map(lambda s: "Cl" in s).values
m_cl = RandomForestRegressor(n_estimators=100, random_state=0,
                             n_jobs=-1).fit(X[~mask_cl], y[~mask_cl])
r2_cl = r2_score(y[mask_cl], m_cl.predict(X[mask_cl]))
print(f"② 留出全部氯化物  : n_test = {mask_cl.sum():4d}   R2 = {r2_cl:.3f}")
print(f"\n差距 = {r2_rand - r2_cl:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4), sharex=True, sharey=True)
for ax, (yt, yp, title, r2) in zip(axes, [
    (y_te, m_rand.predict(X_te), "随机划分", r2_rand),
    (y[mask_cl], m_cl.predict(X[mask_cl]), "留出全部氯化物", r2_cl),
]):
    ax.scatter(yt, yp, s=10, alpha=0.4, color=C.navy, edgecolor="none")
    ax.plot([0, 10], [0, 10], "--", color="#888888", lw=1.2)
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)
    ax.set_xlabel("DFT (PBE) band gap / eV")
    ax.set_title(f"{title}: R2 = {r2:.3f}", fontsize=11)
axes[0].set_ylabel("Predicted band gap / eV")
plt.show()

**讲解**：随机划分的 $R^2$ 在 0.75 上下，留出全部氯化物后掉到 0.5 左右——**差距约 0.25**。

注意这个差距比正文里氟化物那个（$0.83 \to 0.19$）要小。原因值得想一想：

- **氟**是所有元素里电负性最高的，氟化物的带隙分布明显偏高（很多是宽带隙绝缘体），在特征空间里自成一类。训练集里没有氟，模型就完全没见过那个区域；
- **氯**的电负性和离子半径更“居中”，氯化物与溴化物、硫化物在特征空间里有相当的重叠，模型可以从其他卤化物那里借到一些可迁移的知识。

**换句话说，外推有多难，取决于留出的那类化学离训练分布有多远。** 这正是“适用域（applicability domain）”概念的量化表述，也是第二讲 Tanimoto 适用域检查、第四讲高斯过程 $\sigma(x)$ 想解决的同一个问题。

**回答第 5 问：应该相信留出整类化学的那个数字。**

如果你的目的是“用模型预测一类训练集里完全没有的新化合物”，那么随机划分的 $R^2$ 在**定义上**就不是对这个任务的评估——它测的是内插能力，而你要用的是外推能力。

> **📌 一个可操作的建议**：报告模型性能时，同时给出
> 1. 随机划分的 $R^2$（上界，代表内插能力）；
> 2. 若干组留出整类化学的 $R^2$（下界，代表外推能力）；
> 3. 实际使用时，对每个预测检查它的化学体系是否在训练集覆盖范围内。
>
> 只报第 1 个数字，是材料机器学习论文里最常见、也最容易误导人的做法。